In [ ]:
import pandas as pd
import numpy as np

df_analise = pd.read_csv("análise_fornecedores.csv", encoding='latin1', sep=";", low_memory=False)
display(df_analise)

display(df_analise.info())

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao
0,1,05/01/2025,Frigorífico Sul,Picanha,82.0,25,8,9,2.0,9,entrega rápida
1,2,07/01/2025,Carnes Premium,Frango,20.0,40,8,8,3.0,8,NaN
2,3,09/01/2025,Proteína Nobre,Costela,47.0,35,8,7,2.0,8,NaN
3,4,10/01/2025,Frigorifico Sul,Picanha,80.0,20,7,8,2.0,8,erro no nome
4,5,12/01/2025,Distribuidora Boi Forte,Frango,17.0,50,6,7,4.0,6,atraso na entrega
5,6,14/01/2025,Carnes Premium,Picanha,95.0,15,9,9,3.0,9,NaN
6,7,16/01/2025,Proteína Nobre,Frango,19.0,60,7,7,2.0,7,NaN
7,8,18/01/2025,Frigorífico Sul,Costela,45.0,30,8,8,3.0,8,NaN
8,9,20/01/2025,Carnes Premium,Costela,52.0,28,9,9,3.0,9,NaN
9,10,22/01/2025,Proteina Nobre,Picanha,88.0,18,8,8,2.0,8,nome inconsistente


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           40 non-null     int64  
 1   data_pedido         40 non-null     object 
 2   fornecedor          40 non-null     object 
 3   produto             40 non-null     object 
 4   preco_kg            39 non-null     float64
 5   quantidade_kg       40 non-null     int64  
 6   qualidade           40 non-null     int64  
 7   aceitacao_socios    40 non-null     int64  
 8   tempo_entrega_dias  39 non-null     float64
 9   avaliacao_chef      40 non-null     int64  
 10  observacao          8 non-null      object 
dtypes: float64(2), int64(5), object(4)
memory usage: 3.6+ KB


None

Ao observar as infomações acima, é possível identificar algun erros:

1. Na coluna "data_pedido" temos o tipo informado como object= string. O correto, neste caso, é o farmato "datetime" para realizarmos análises temporais.

1. Não padronização de letras ( maiúsculas e minuscula) e acentuações.  

1.  Na coluna "preco_kg" temos um valor negativo e outro nulo.

2. Na coluna "tempo_entrega_dias" temos um valor nulo, o que faz o pandas interpretar como sendo do tipo float. Ao fazer o tratamento deste valor nulo, podemos converter para o tipo int, já que estamos tratando de dias.

2. A coluna "observacao" possui 32 valores nulos, o que indica que a maioria dos registros não possuem comentários.Precisamos então preencher com algum dado.


In [ ]:
#Tranformando a coluna data para o formato datetime
#Colocando essas datas dos meses mais antigos para os meses mais recentes (analisar evolução)

df_analise["data_pedido"] = pd.to_datetime(df_analise["data_pedido"], format="%d/%m/%Y")
df_analise = df_analise.sort_values(by="data_pedido", ascending=True)

#Tratamento de padronização capitalize

df_analise["fornecedor"] = df_analise["fornecedor"].str.strip().str.title()

import unicodedata

def remover_acentos(texto):
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("ASCII")

df_analise["fornecedor"] = df_analise["fornecedor"].apply(remover_acentos)

df_analise["fornecedor"].value_counts()

#Tratando o valor nulo da coluna "preco_kg"

df_analise[(df_analise["fornecedor"] == "Carnes premium") & (df_analise["produto"] == "Frango")]

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao


A coluna "preco_kg" apresenta um valor nulo. Para preenchermos, podemos realizar uma estimativa com base nos valores do mesmo produto e fornecedor.

In [ ]:
df_analise[(df_analise["fornecedor"] == "Carnes Premium") & (df_analise["produto"] == "Frango")]["preco_kg"].mean()

np.float64(20.333333333333332)

Calculamos uma média de quanto seria este valor com base nos outros valores do mesmo fornecedor e do mesmo produto.

In [ ]:
df_analise.loc[
    (df_analise["fornecedor"] == "Carnes premium") &
    (df_analise["produto"] == "Frango") &
    (df_analise["preco_kg"].isna()),
    "preco_kg"
] = 20.33
display(df_analise["preco_kg"].unique())

array([ 82.,  20.,  47.,  80.,  17.,  95.,  19.,  45.,  52.,  88.,  79.,
        nan,  18.,  96., -80.,  51.,  44.,  94.,  78.,  21.,  46.,  53.,
        83., 300.,  89.,  40.])

O preço apareceu na coluna e agora precisamos tratar o próximo dado que está com um valor negativo na coluna "peco_kg".


In [ ]:
df_analise.loc[df_analise["preco_kg"] < 0, "preco_kg"] = None
display(df_analise["preco_kg"].unique())

array([ 82.,  20.,  47.,  80.,  17.,  95.,  19.,  45.,  52.,  88.,  79.,
        nan,  18.,  96.,  51.,  44.,  94.,  78.,  21.,  46.,  53.,  83.,
       300.,  89.,  40.])

In [ ]:
display(df_analise[["id_pedido", "fornecedor",	"produto",	"preco_kg",	"quantidade_kg"]])

,id_pedido,fornecedor,produto,preco_kg,quantidade_kg
0,1,Frigorifico Sul,Picanha,82.0,25
1,2,Carnes Premium,Frango,20.0,40
2,3,Proteina Nobre,Costela,47.0,35
3,4,Frigorifico Sul,Picanha,80.0,20
4,5,Distribuidora Boi Forte,Frango,17.0,50
5,6,Carnes Premium,Picanha,95.0,15
6,7,Proteina Nobre,Frango,19.0,60
7,8,Frigorifico Sul,Costela,45.0,30
8,9,Carnes Premium,Costela,52.0,28
9,10,Proteina Nobre,Picanha,88.0,18


Deixamos a linha vazia porque assim aplicamos a limpeza desse dados para em seguida podermos realizar uma estimativa de preço com base nos outros preços co mesmo produto e mesmo fornecedor.


In [ ]:
df_id_pedido_17= df_analise[df_analise["id_pedido"]==17]
display(df_id_pedido_17)

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao
16,17,2025-02-02,Frigorifico Sul,Picanha,NaN,22,8,8,2.0,8,preço negativo


Acima criamos uma condicional a qual recebe o valor da nossa antiga base dados e avalia se a coluna id_pedido é exatamente igual ao id do pedido que queremos selecionar. Dessa forma podemos realizar uma visualização e análise melhor.


In [ ]:
df_analise[(df_analise["fornecedor"] == "Frigorifico Sul") & (df_analise["produto"] == "Picanha")]["preco_kg"].mean()

np.float64(81.66666666666667)

Descobrimos o valor estimado de acordo com o mesmo fornecedor e o mesmo tipo de produto,calculamos a média e agorapodemos implementar na nossa base de dados.

In [ ]:
df_analise.loc[
    (df_analise["fornecedor"] == "Frigorifico Sul") &
    (df_analise["produto"] == "Picanha") &
    (df_analise["preco_kg"].isna()),
    "preco_kg"
] = 81.66
display(df_analise["preco_kg"].unique())

array([ 82.  ,  20.  ,  47.  ,  80.  ,  17.  ,  95.  ,  19.  ,  45.  ,
        52.  ,  88.  ,  79.  ,    nan,  18.  ,  96.  ,  81.66,  51.  ,
        44.  ,  94.  ,  78.  ,  21.  ,  46.  ,  53.  ,  83.  , 300.  ,
        89.  ,  40.  ])